# xView2 Damage Segmentation Training Notebook

This notebook is structured like a more typical deep learning training notebook:

1. imports and config
2. dataset indexing and split
3. dataset and augmentations
4. quick visualization
5. model
6. loss and metrics
7. train/validate loops
8. training run
9. test evaluation and plots


In [ ]:
# ============================================================
# 1. Imports and config
# ============================================================

import csv
import json
import random
import sys
import time
from pathlib import Path

import cv2
import matplotlib
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from shapely import wkt
from skimage.draw import polygon as draw_polygon
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, Dataset
from torchvision import models

cv2.setNumThreads(0)

# ── Point this to a config.json to load experiment settings ──
# e.g. "training_runs/siamese_resunet_xview2_3/config.json"
CONFIG_PATH = None

DEFAULTS = {
    "data_root": "3/train/train",
    "seed": 42,
    "output_root": "training_runs",
    "experiment_name": "siamese_resunet_xview2",
    "resume_training": True,
    "train_ratio": 0.8,
    "val_ratio": 0.1,
    "test_ratio": 0.1,
    "image_size": [512, 512],
    "num_classes": 5,
    "backbone": "resnet34",
    "pretrained": True,
    "decoder_channels": [256, 128, 64, 32],
    "batch_size": 2,
    "num_workers": 0,
    "epochs": 50,
    "lr": 3e-4,
    "encoder_lr": 1e-5,
    "weight_decay": 0.05,
    "grad_clip": 1.0,
    "accumulation_steps": 4,
    "use_amp": True,
    "scheduler": "cosine",
    "min_lr": 1e-6,
    "ce_weight": 1.0,
    "dice_weight": 1.0,
    "label_smoothing": 0.05,
    "class_weight_method": "log_inverse",
    "class_weight_samples": None,
    "freeze_epochs": 15,
    "early_stopping_patience": 10,
    "run_training": False,
    "run_smoke_test": True,
    "show_examples": 3,
    "aug_hflip": 0.5,
    "aug_vflip": 0.2,
    "aug_rotate90": 0.5,
    "aug_brightness": 0.12,
    "aug_contrast": 0.12,
    "aug_saturation": 0.08,
    "aug_blur_prob": 0.10,
    "aug_noise_prob": 0.10,
}

if CONFIG_PATH and Path(CONFIG_PATH).exists():
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        file_config = json.load(f)
    CONFIG = {**DEFAULTS, **file_config}
    print(f"Loaded config from {CONFIG_PATH}")
else:
    CONFIG = dict(DEFAULTS)
    if CONFIG_PATH:
        print(f"WARNING: {CONFIG_PATH} not found, using defaults")
    else:
        print("Using default config (set CONFIG_PATH to load from file)")

DAMAGE_CLASSES = {
    0: "Background",
    1: "No Damage",
    2: "Minor Damage",
    3: "Major Damage",
    4: "Destroyed",
}
DAMAGE_COLORS = ["black", "green", "yellow", "orange", "red"]
DAMAGE_CMAP = ListedColormap(DAMAGE_COLORS)
DAMAGE_LEGEND = [
    Patch(facecolor=color, edgecolor="white", label=DAMAGE_CLASSES[i])
    for i, color in enumerate(DAMAGE_COLORS)
]
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

RUN_DIR = Path(CONFIG["output_root"]) / CONFIG["experiment_name"]
RUN_DIR.mkdir(parents=True, exist_ok=True)
CONFIG["run_dir"] = str(RUN_DIR)
CONFIG["best_checkpoint_path"] = str(RUN_DIR / "best_model.pt")
CONFIG["last_checkpoint_path"] = str(RUN_DIR / "last_model.pt")
CONFIG["history_json_path"] = str(RUN_DIR / "history.json")
CONFIG["metrics_csv_path"] = str(RUN_DIR / "metrics.csv")
CONFIG["config_json_path"] = str(RUN_DIR / "config.json")

with open(CONFIG["config_json_path"], "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2)

print(json.dumps(CONFIG, indent=2))


In [ ]:
# ============================================================
# 2. Setup, indexing, and train/val/test split
# ============================================================

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def build_sample_ids(data_root):
    data_root = Path(data_root)
    images_dir = data_root / "images"
    labels_dir = data_root / "labels"
    sample_ids = []

    for pre_path in sorted(images_dir.glob("*_pre_disaster.png")):
        sample_id = pre_path.name.replace("_pre_disaster.png", "")
        required = [
            images_dir / f"{sample_id}_pre_disaster.png",
            images_dir / f"{sample_id}_post_disaster.png",
            labels_dir / f"{sample_id}_pre_disaster.json",
            labels_dir / f"{sample_id}_post_disaster.json",
        ]
        if all(path.exists() for path in required):
            sample_ids.append(sample_id)
    return sample_ids


def split_sample_ids(sample_ids, train_ratio, val_ratio, test_ratio, seed):
    sample_ids = list(sample_ids)
    random.Random(seed).shuffle(sample_ids)
    total = len(sample_ids)
    train_count = int(round(total * train_ratio))
    val_count = int(round(total * val_ratio))
    test_count = total - train_count - val_count

    train_ids = sample_ids[:train_count]
    val_ids = sample_ids[train_count:train_count + val_count]
    test_ids = sample_ids[train_count + val_count:train_count + val_count + test_count]

    assert set(train_ids).isdisjoint(set(val_ids))
    assert set(train_ids).isdisjoint(set(test_ids))
    assert set(val_ids).isdisjoint(set(test_ids))
    return train_ids, val_ids, test_ids


seed_everything(CONFIG["seed"])
DEVICE = get_device()
SAMPLE_IDS = build_sample_ids(CONFIG["data_root"])
TRAIN_IDS, VAL_IDS, TEST_IDS = split_sample_ids(
    SAMPLE_IDS,
    CONFIG["train_ratio"],
    CONFIG["val_ratio"],
    CONFIG["test_ratio"],
    CONFIG["seed"],
)

print("Device:", DEVICE)
print("Total pairs:", len(SAMPLE_IDS))
print("Train:", len(TRAIN_IDS))
print("Val:", len(VAL_IDS))
print("Test:", len(TEST_IDS))


In [ ]:
# ============================================================
# 3. Dataset, masks, and augmentations
# ============================================================

DAMAGE_MAP = {
    "no-damage": 1,
    "minor-damage": 2,
    "major-damage": 3,
    "destroyed": 4,
    "un-classified": 1,
}


def polygon_list(geometry):
    if geometry.is_empty:
        return []
    if geometry.geom_type == "Polygon":
        return [geometry]
    if geometry.geom_type == "MultiPolygon":
        return list(geometry.geoms)
    if geometry.geom_type == "GeometryCollection":
        polygons = []
        for item in geometry.geoms:
            polygons.extend(polygon_list(item))
        return polygons
    return []


def rasterize_mask(label_path, image_shape, is_post):
    h, w = image_shape[:2]
    mask = np.zeros((h, w), dtype=np.int64)
    with open(label_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    for obj in data["features"]["xy"]:
        geometry = wkt.loads(obj["wkt"])
        for poly in polygon_list(geometry):
            if poly.is_empty or poly.area < 1:
                continue
            coords = np.asarray(poly.exterior.coords)
            if coords.shape[0] < 3:
                continue
            rr, cc = draw_polygon(coords[:, 1], coords[:, 0], shape=(h, w))
            if rr.size == 0:
                continue
            if is_post:
                subtype = obj["properties"].get("subtype", "un-classified")
                mask[rr, cc] = DAMAGE_MAP.get(subtype, 1)
            else:
                mask[rr, cc] = 1
    return mask


def color_jitter(img):
    img = img.astype(np.float32) / 255.0
    b = 1.0 + random.uniform(-CONFIG["aug_brightness"], CONFIG["aug_brightness"])
    c = 1.0 + random.uniform(-CONFIG["aug_contrast"], CONFIG["aug_contrast"])
    s = 1.0 + random.uniform(-CONFIG["aug_saturation"], CONFIG["aug_saturation"])
    img = np.clip(img * b, 0.0, 1.0)
    mean = img.mean(axis=(0, 1), keepdims=True)
    img = np.clip((img - mean) * c + mean, 0.0, 1.0)
    gray = img.mean(axis=2, keepdims=True)
    img = np.clip(gray + s * (img - gray), 0.0, 1.0)
    return (img * 255.0).astype(np.uint8)


def apply_train_augmentations(pre_img, post_img, pre_mask, post_mask):
    if random.random() < CONFIG["aug_hflip"]:
        pre_img = np.flip(pre_img, axis=1).copy()
        post_img = np.flip(post_img, axis=1).copy()
        pre_mask = np.flip(pre_mask, axis=1).copy()
        post_mask = np.flip(post_mask, axis=1).copy()

    if random.random() < CONFIG["aug_vflip"]:
        pre_img = np.flip(pre_img, axis=0).copy()
        post_img = np.flip(post_img, axis=0).copy()
        pre_mask = np.flip(pre_mask, axis=0).copy()
        post_mask = np.flip(post_mask, axis=0).copy()

    if random.random() < CONFIG["aug_rotate90"]:
        k = random.choice([1, 2, 3])
        pre_img = np.rot90(pre_img, k=k).copy()
        post_img = np.rot90(post_img, k=k).copy()
        pre_mask = np.rot90(pre_mask, k=k).copy()
        post_mask = np.rot90(post_mask, k=k).copy()

    pre_img = color_jitter(pre_img)
    post_img = color_jitter(post_img)

    if random.random() < CONFIG["aug_blur_prob"]:
        pre_img = cv2.GaussianBlur(pre_img, (3, 3), 0)
    if random.random() < CONFIG["aug_blur_prob"]:
        post_img = cv2.GaussianBlur(post_img, (3, 3), 0)

    if random.random() < CONFIG["aug_noise_prob"]:
        noise = np.random.normal(0, 8, size=pre_img.shape).astype(np.float32)
        pre_img = np.clip(pre_img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    if random.random() < CONFIG["aug_noise_prob"]:
        noise = np.random.normal(0, 8, size=post_img.shape).astype(np.float32)
        post_img = np.clip(post_img.astype(np.float32) + noise, 0, 255).astype(np.uint8)

    return pre_img, post_img, pre_mask, post_mask


class XView2Dataset(Dataset):
    def __init__(self, data_root, sample_ids, train=False):
        self.data_root = Path(data_root)
        self.images_dir = self.data_root / "images"
        self.labels_dir = self.data_root / "labels"
        self.sample_ids = list(sample_ids)
        self.train = train

    def __len__(self):
        return len(self.sample_ids)

    def load_image(self, path):
        image = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if image is None:
            raise FileNotFoundError(path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        return image

    def resize_image(self, image):
        target_h, target_w = CONFIG["image_size"]
        if image.shape[:2] == (target_h, target_w):
            return image
        return cv2.resize(image, (target_w, target_h), interpolation=cv2.INTER_LINEAR)

    def resize_mask(self, mask):
        target_h, target_w = CONFIG["image_size"]
        if mask.shape[:2] == (target_h, target_w):
            return mask
        return cv2.resize(mask.astype(np.uint8), (target_w, target_h), interpolation=cv2.INTER_NEAREST).astype(np.int64)

    def normalize(self, image):
        image = image.astype(np.float32) / 255.0
        image = (image - IMAGENET_MEAN) / IMAGENET_STD
        return torch.from_numpy(image).permute(2, 0, 1).float()

    def __getitem__(self, idx):
        sample_id = self.sample_ids[idx]
        pre_img = self.load_image(self.images_dir / f"{sample_id}_pre_disaster.png")
        post_img = self.load_image(self.images_dir / f"{sample_id}_post_disaster.png")
        pre_mask = rasterize_mask(self.labels_dir / f"{sample_id}_pre_disaster.json", pre_img.shape, False)
        post_mask = rasterize_mask(self.labels_dir / f"{sample_id}_post_disaster.json", post_img.shape, True)

        pre_img = self.resize_image(pre_img)
        post_img = self.resize_image(post_img)
        pre_mask = self.resize_mask(pre_mask)
        post_mask = self.resize_mask(post_mask)

        if self.train:
            pre_img, post_img, pre_mask, post_mask = apply_train_augmentations(pre_img, post_img, pre_mask, post_mask)

        return {
            "sample_id": sample_id,
            "pre_image": self.normalize(pre_img),
            "post_image": self.normalize(post_img),
            "pre_mask": torch.from_numpy(pre_mask).long(),
            "post_mask": torch.from_numpy(post_mask).long(),
        }


train_dataset = XView2Dataset(CONFIG["data_root"], TRAIN_IDS, train=True)
val_dataset = XView2Dataset(CONFIG["data_root"], VAL_IDS, train=False)
test_dataset = XView2Dataset(CONFIG["data_root"], TEST_IDS, train=False)

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=CONFIG["num_workers"])
val_loader = DataLoader(val_dataset, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"])
test_loader = DataLoader(test_dataset, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"])


In [ ]:
# ============================================================
# 4. Quick data check
# ============================================================

def denormalize_image(tensor):
    image = tensor.detach().cpu().permute(1, 2, 0).numpy()
    image = (image * IMAGENET_STD) + IMAGENET_MEAN
    return np.clip(image, 0.0, 1.0)


batch = next(iter(train_loader))
print("Sample id:", batch["sample_id"][0])
print("Pre image:", tuple(batch["pre_image"].shape))
print("Post image:", tuple(batch["post_image"].shape))
print("Pre mask:", tuple(batch["pre_mask"].shape), torch.unique(batch["pre_mask"]))
print("Post mask:", tuple(batch["post_mask"].shape), torch.unique(batch["post_mask"]))

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
axes[0, 0].imshow(denormalize_image(batch["pre_image"][0]))
axes[0, 0].set_title("Pre image")
axes[0, 0].axis("off")

axes[0, 1].imshow(batch["pre_mask"][0].cpu(), cmap="gray")
axes[0, 1].set_title("Pre mask")
axes[0, 1].axis("off")

axes[1, 0].imshow(denormalize_image(batch["post_image"][0]))
axes[1, 0].set_title("Post image")
axes[1, 0].axis("off")

axes[1, 1].imshow(batch["post_mask"][0].cpu(), cmap=DAMAGE_CMAP, vmin=0, vmax=4)
axes[1, 1].set_title("Post damage mask")
axes[1, 1].axis("off")
axes[1, 1].legend(handles=DAMAGE_LEGEND, loc="upper left", fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 5. Siamese ResNet U-Net model
# ============================================================

RESNET_CHANNELS = {
    "resnet18": [64, 64, 128, 256, 512],
    "resnet34": [64, 64, 128, 256, 512],
    "resnet50": [64, 256, 512, 1024, 2048],
    "resnet101": [64, 256, 512, 1024, 2048],
}


def build_backbone(name, pretrained=True):
    weights_map = {
        "resnet18": models.ResNet18_Weights.DEFAULT,
        "resnet34": models.ResNet34_Weights.DEFAULT,
        "resnet50": models.ResNet50_Weights.DEFAULT,
        "resnet101": models.ResNet101_Weights.DEFAULT,
    }
    constructor = getattr(models, name)
    if not pretrained:
        return constructor(weights=None)
    try:
        return constructor(weights=weights_map[name])
    except Exception as exc:
        print(f"Could not load pretrained weights, using random init instead: {exc}")
        return constructor(weights=None)


class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class ResNetEncoder(nn.Module):
    def __init__(self, backbone_name="resnet34", pretrained=True):
        super().__init__()
        backbone = build_backbone(backbone_name, pretrained=pretrained)
        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)
        self.maxpool = backbone.maxpool
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4
        self.channels = RESNET_CHANNELS[backbone_name]

    def forward(self, x):
        x0 = self.stem(x)
        x1 = self.layer1(self.maxpool(x0))
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)
        return [x0, x1, x2, x3, x4]


class FusionBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = ConvBlock(channels * 3, channels)

    def forward(self, pre_feat, post_feat):
        diff = torch.abs(post_feat - pre_feat)
        x = torch.cat([pre_feat, post_feat, diff], dim=1)
        return self.block(x)


class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.block = ConvBlock(in_channels + skip_channels, out_channels)

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.block(x)


class SiameseResUNet(nn.Module):
    def __init__(self, backbone="resnet34", num_classes=5, pretrained=True):
        super().__init__()
        self.encoder = ResNetEncoder(backbone, pretrained=pretrained)
        enc = self.encoder.channels
        dec = CONFIG["decoder_channels"]

        self.fuse0 = FusionBlock(enc[0])
        self.fuse1 = FusionBlock(enc[1])
        self.fuse2 = FusionBlock(enc[2])
        self.fuse3 = FusionBlock(enc[3])
        self.fuse4 = FusionBlock(enc[4])

        self.bottleneck = ConvBlock(enc[4], dec[0])
        self.dec4 = DecoderBlock(dec[0], enc[3], dec[0])
        self.dec3 = DecoderBlock(dec[0], enc[2], dec[1])
        self.dec2 = DecoderBlock(dec[1], enc[1], dec[2])
        self.dec1 = DecoderBlock(dec[2], enc[0], dec[3])
        self.head = nn.Conv2d(dec[3], num_classes, kernel_size=1)

    def freeze_encoder(self):
        for p in self.encoder.parameters():
            p.requires_grad = False
        n = sum(p.numel() for p in self.encoder.parameters())
        print(f"Encoder FROZEN ({n:,} params)")

    def unfreeze_encoder(self, n_layers=2):
        layers = [self.encoder.stem, self.encoder.layer1, self.encoder.layer2,
                  self.encoder.layer3, self.encoder.layer4]
        for layer in layers[-n_layers:]:
            for p in layer.parameters():
                p.requires_grad = True
        for p in self.encoder.stem[1].parameters():
            p.requires_grad = True
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Unfroze last {n_layers} encoder layers — trainable: {trainable:,} params")

    def forward(self, pre_img, post_img):
        out_size = pre_img.shape[-2:]
        pre = self.encoder(pre_img)
        post = self.encoder(post_img)

        f0 = self.fuse0(pre[0], post[0])
        f1 = self.fuse1(pre[1], post[1])
        f2 = self.fuse2(pre[2], post[2])
        f3 = self.fuse3(pre[3], post[3])
        f4 = self.fuse4(pre[4], post[4])

        x = self.bottleneck(f4)
        x = self.dec4(x, f3)
        x = self.dec3(x, f2)
        x = self.dec2(x, f1)
        x = self.dec1(x, f0)
        x = self.head(x)
        x = F.interpolate(x, size=out_size, mode="bilinear", align_corners=False)
        return x


model = SiameseResUNet(
    backbone=CONFIG["backbone"],
    num_classes=CONFIG["num_classes"],
    pretrained=CONFIG["pretrained"],
).to(DEVICE)

with torch.no_grad():
    test_logits = model(batch["pre_image"].to(DEVICE), batch["post_image"].to(DEVICE))
print("Model output:", tuple(test_logits.shape))
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")


In [ ]:
# ============================================================
# 6. Loss and metrics
# ============================================================

def compute_class_weights(dataset, num_classes=5, method="log_inverse", max_samples=None):
    counts = torch.zeros(num_classes, dtype=torch.float64)
    total = len(dataset) if max_samples is None else min(len(dataset), max_samples)
    print(f"Computing class weights from {total} samples...")
    for i in range(total):
        sample = dataset[i]
        counts += torch.bincount(sample["post_mask"].view(-1), minlength=num_classes).double()

    print("Pixel counts per class:")
    for i in range(num_classes):
        print(f"  {DAMAGE_CLASSES[i]:<12} {int(counts[i]):>12,}")

    if method == "log_inverse":
        total_px = counts.sum()
        log_w = torch.log(total_px / counts.clamp_min(1.0))
        weights = torch.clamp(log_w / log_w.min(), 1.0, 20.0)
    else:
        weights = counts.sum() / counts.clamp_min(1.0)
        weights = weights / weights.mean()

    print("Class weights:")
    for i in range(num_classes):
        print(f"  {DAMAGE_CLASSES[i]:<12} {float(weights[i]):>6.2f}")
    return weights.float()


class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1)
        one_hot = F.one_hot(targets, num_classes=logits.shape[1]).permute(0, 3, 1, 2).float()
        intersection = (probs * one_hot).sum(dim=(0, 2, 3))
        union = probs.sum(dim=(0, 2, 3)) + one_hot.sum(dim=(0, 2, 3))
        dice = (2 * intersection + self.smooth) / (union + self.smooth)
        return 1.0 - dice.mean()


class CombinedLoss(nn.Module):
    def __init__(self, class_weights=None, label_smoothing=0.0):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=label_smoothing)
        self.dice = DiceLoss()

    def forward(self, logits, targets):
        ce = self.ce(logits, targets)
        dice = self.dice(logits, targets)
        return CONFIG["ce_weight"] * ce + CONFIG["dice_weight"] * dice


def confusion_matrix_from_logits(logits, targets, num_classes):
    preds = logits.argmax(dim=1).view(-1).cpu()
    targets = targets.view(-1).cpu()
    encoded = targets * num_classes + preds
    matrix = torch.bincount(encoded, minlength=num_classes * num_classes)
    return matrix.view(num_classes, num_classes)


def metrics_from_confusion(conf_matrix):
    conf = conf_matrix.float()
    tp = conf.diag()
    fp = conf.sum(dim=0) - tp
    fn = conf.sum(dim=1) - tp
    pixel_acc = (tp.sum() / conf.sum().clamp_min(1.0)).item()
    per_class_iou = tp / (tp + fp + fn).clamp_min(1.0)
    per_class_dice = (2 * tp) / (2 * tp + fp + fn).clamp_min(1.0)

    precision = tp / (tp + fp).clamp_min(1.0)
    recall = tp / (tp + fn).clamp_min(1.0)
    per_class_f1 = (2 * precision * recall) / (precision + recall).clamp_min(1e-8)

    damage_f1 = per_class_f1[1:].mean().item()

    return {
        "pixel_accuracy": pixel_acc,
        "per_class_iou": per_class_iou,
        "miou": per_class_iou.mean().item(),
        "per_class_dice": per_class_dice,
        "mean_dice": per_class_dice.mean().item(),
        "per_class_f1": per_class_f1,
        "damage_macro_f1": damage_f1,
        "confusion_matrix": conf_matrix,
    }


class_weights = compute_class_weights(
    train_dataset,
    num_classes=CONFIG["num_classes"],
    method=CONFIG.get("class_weight_method", "frequency"),
    max_samples=CONFIG.get("class_weight_samples"),
)
criterion = CombinedLoss(
    class_weights=class_weights.to(DEVICE),
    label_smoothing=CONFIG.get("label_smoothing", 0.0),
)
print(f"Label smoothing: {CONFIG.get('label_smoothing', 0.0)}")


In [ ]:
# ============================================================
# 7. Training and validation loops
# ============================================================

scaler = torch.cuda.amp.GradScaler(enabled=CONFIG["use_amp"] and DEVICE.type == "cuda")


def amp_context():
    if CONFIG["use_amp"] and DEVICE.type == "cuda":
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    from contextlib import nullcontext
    return nullcontext()


def make_optimizer(model, phase):
    if phase == 1:
        params = [p for p in model.parameters() if p.requires_grad]
        opt = torch.optim.AdamW(params, lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
        print(f"Phase 1 optimizer: all trainable params, lr={CONFIG['lr']}")
    else:
        encoder_params = list(model.encoder.parameters())
        encoder_ids = {id(p) for p in encoder_params}
        decoder_params = [p for p in model.parameters() if id(p) not in encoder_ids]
        trainable_encoder = [p for p in encoder_params if p.requires_grad]
        opt = torch.optim.AdamW([
            {"params": trainable_encoder, "lr": CONFIG["encoder_lr"]},
            {"params": decoder_params, "lr": CONFIG["lr"]},
        ], weight_decay=CONFIG["weight_decay"])
        print(f"Phase 2 optimizer: encoder lr={CONFIG['encoder_lr']}, decoder lr={CONFIG['lr']}")
    return opt


def make_scheduler(optimizer, remaining_epochs):
    return torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(remaining_epochs, 1), eta_min=CONFIG["min_lr"]
    )


def train_one_epoch(model, loader, optimizer, scheduler=None):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    total_loss = 0.0
    total_conf = torch.zeros(CONFIG["num_classes"], CONFIG["num_classes"], dtype=torch.int64)

    for step, batch in enumerate(loader, start=1):
        pre_img = batch["pre_image"].to(DEVICE)
        post_img = batch["post_image"].to(DEVICE)
        targets = batch["post_mask"].to(DEVICE)

        with amp_context():
            logits = model(pre_img, post_img)
            loss = criterion(logits, targets)
            scaled_loss = loss / CONFIG["accumulation_steps"]

        if scaler.is_enabled():
            scaler.scale(scaled_loss).backward()
        else:
            scaled_loss.backward()

        if step % CONFIG["accumulation_steps"] == 0 or step == len(loader):
            if CONFIG["grad_clip"] is not None:
                if scaler.is_enabled():
                    scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])

            if scaler.is_enabled():
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item()
        total_conf += confusion_matrix_from_logits(logits, targets, CONFIG["num_classes"])

    metrics = metrics_from_confusion(total_conf)
    metrics["loss"] = total_loss / max(len(loader), 1)
    return metrics


def validate_one_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total_conf = torch.zeros(CONFIG["num_classes"], CONFIG["num_classes"], dtype=torch.int64)

    with torch.no_grad():
        for batch in loader:
            pre_img = batch["pre_image"].to(DEVICE)
            post_img = batch["post_image"].to(DEVICE)
            targets = batch["post_mask"].to(DEVICE)

            with amp_context():
                logits = model(pre_img, post_img)
                loss = criterion(logits, targets)

            total_loss += loss.item()
            total_conf += confusion_matrix_from_logits(logits, targets, CONFIG["num_classes"])

    metrics = metrics_from_confusion(total_conf)
    metrics["loss"] = total_loss / max(len(loader), 1)
    return metrics


def run_smoke_test():
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    optimizer.zero_grad(set_to_none=True)
    small_batch = next(iter(train_loader))
    pre_img = small_batch["pre_image"].to(DEVICE)
    post_img = small_batch["post_image"].to(DEVICE)
    targets = small_batch["post_mask"].to(DEVICE)

    logits = model(pre_img, post_img)
    loss = criterion(logits, targets)
    loss.backward()
    optimizer.zero_grad(set_to_none=True)
    print(f"Smoke test passed. One forward/backward pass worked. Loss = {loss.item():.4f}")


def checkpoint_payload(epoch, best_f1, best_miou, history, optimizer, scheduler):
    payload = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_damage_f1": best_f1,
        "best_miou": best_miou,
        "history": history,
        "config": CONFIG,
    }
    if scheduler is not None:
        payload["scheduler_state_dict"] = scheduler.state_dict()
    return payload


def save_history(history):
    with open(CONFIG["history_json_path"], "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)


def append_metrics_csv(row):
    csv_path = Path(CONFIG["metrics_csv_path"])
    write_header = not csv_path.exists()
    with open(csv_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if write_header:
            writer.writeheader()
        writer.writerow(row)


def load_last_checkpoint_if_available():
    checkpoint_path = Path(CONFIG["last_checkpoint_path"])
    if not CONFIG["resume_training"] or not checkpoint_path.exists():
        return 0, -1.0, -1.0, []

    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])

    start_epoch = checkpoint.get("epoch", -1) + 1
    best_f1 = checkpoint.get("best_damage_f1", -1.0)
    best_miou = checkpoint.get("best_miou", -1.0)
    history = checkpoint.get("history", [])
    print(f"Resumed from {checkpoint_path} at epoch {start_epoch}")
    print(f"  best_damage_f1={best_f1:.4f}, best_miou={best_miou:.4f}")
    return start_epoch, best_f1, best_miou, history


if CONFIG["run_smoke_test"]:
    run_smoke_test()


In [ ]:
# ============================================================
# 8. Training run (two-phase + early stopping)
# ============================================================

start_epoch, best_f1, best_miou, history = load_last_checkpoint_if_available()
if not isinstance(history, list):
    history = []

freeze_epochs = CONFIG.get("freeze_epochs", 0)
patience = CONFIG.get("early_stopping_patience", 10)

if CONFIG["run_training"]:
    # ── Determine current phase from start_epoch ──
    current_phase = 1 if (freeze_epochs > 0 and start_epoch < freeze_epochs) else 2

    if current_phase == 1:
        model.freeze_encoder()
    else:
        model.unfreeze_encoder(n_layers=2)

    optimizer = make_optimizer(model, current_phase)

    if current_phase == 1:
        remaining = freeze_epochs - start_epoch
    else:
        remaining = CONFIG["epochs"] - start_epoch
    scheduler = make_scheduler(optimizer, remaining)

    # Fast-forward scheduler if resuming mid-phase
    if current_phase == 1:
        for _ in range(start_epoch):
            scheduler.step()
    else:
        for _ in range(start_epoch - freeze_epochs):
            scheduler.step()

    patience_counter = 0

    print("=" * 70)
    if freeze_epochs > 0:
        print(f"PHASE {current_phase} | freeze_epochs={freeze_epochs} | "
              f"total_epochs={CONFIG['epochs']} | patience={patience}")
    else:
        print(f"TRAINING | epochs={CONFIG['epochs']} | patience={patience}")
    print("=" * 70)

    for epoch in range(start_epoch, CONFIG["epochs"]):
        # ── Phase transition ──
        if freeze_epochs > 0 and epoch == freeze_epochs and current_phase == 1:
            print()
            print("=" * 70)
            print("PHASE 2 — Unfreezing encoder with differential LR")
            print("=" * 70)
            model.unfreeze_encoder(n_layers=2)
            current_phase = 2
            optimizer = make_optimizer(model, 2)
            scheduler = make_scheduler(optimizer, CONFIG["epochs"] - freeze_epochs)
            patience_counter = 0

        start = time.time()
        train_metrics = train_one_epoch(model, train_loader, optimizer)
        val_metrics = validate_one_epoch(model, val_loader, criterion)
        scheduler.step()

        # ── Build epoch row with F1 metrics ──
        train_f1 = train_metrics["per_class_f1"]
        val_f1 = val_metrics["per_class_f1"]
        epoch_row = {
            "epoch": epoch + 1,
            "phase": current_phase,
            "train_loss": float(train_metrics["loss"]),
            "val_loss": float(val_metrics["loss"]),
            "train_miou": float(train_metrics["miou"]),
            "val_miou": float(val_metrics["miou"]),
            "train_damage_f1": float(train_metrics["damage_macro_f1"]),
            "val_damage_f1": float(val_metrics["damage_macro_f1"]),
            "train_pixel_accuracy": float(train_metrics["pixel_accuracy"]),
            "val_pixel_accuracy": float(val_metrics["pixel_accuracy"]),
            "train_mean_dice": float(train_metrics["mean_dice"]),
            "val_mean_dice": float(val_metrics["mean_dice"]),
            "val_f1_nodmg": float(val_f1[1]),
            "val_f1_minor": float(val_f1[2]),
            "val_f1_major": float(val_f1[3]),
            "val_f1_destroyed": float(val_f1[4]),
            "lr": float(optimizer.param_groups[0]["lr"]),
            "epoch_time_sec": float(time.time() - start),
        }
        history.append(epoch_row)
        save_history(history)
        append_metrics_csv(epoch_row)

        # ── Checkpointing on best damage F1 ──
        torch.save(
            checkpoint_payload(epoch, best_f1, best_miou, history, optimizer, scheduler),
            CONFIG["last_checkpoint_path"],
        )

        improved = False
        if val_metrics["damage_macro_f1"] > best_f1:
            best_f1 = val_metrics["damage_macro_f1"]
            improved = True
        if val_metrics["miou"] > best_miou:
            best_miou = val_metrics["miou"]
            improved = True

        if improved:
            torch.save(
                checkpoint_payload(epoch, best_f1, best_miou, history, optimizer, scheduler),
                CONFIG["best_checkpoint_path"],
            )
            patience_counter = 0
        else:
            patience_counter += 1

        mark = " << best" if improved else ""
        print(
            f"Ep {epoch + 1:>3}/{CONFIG['epochs']} P{current_phase} | "
            f"loss={train_metrics['loss']:.4f}/{val_metrics['loss']:.4f} | "
            f"dmg_F1={val_metrics['damage_macro_f1']:.4f} | "
            f"mIoU={val_metrics['miou']:.4f} | "
            f"NoDmg={float(val_f1[1]):.3f} Min={float(val_f1[2]):.3f} "
            f"Maj={float(val_f1[3]):.3f} Des={float(val_f1[4]):.3f} | "
            f"lr={optimizer.param_groups[0]['lr']:.2e} | "
            f"t={epoch_row['epoch_time_sec']:.0f}s{mark}"
        )

        # ── Early stopping ──
        if patience_counter >= patience:
            if current_phase == 1 and epoch + 1 < freeze_epochs:
                print(f"\nEarly stopping Phase 1 at epoch {epoch + 1} (patience={patience})")
                print("Advancing to Phase 2...\n")
                model.unfreeze_encoder(n_layers=2)
                current_phase = 2
                optimizer = make_optimizer(model, 2)
                scheduler = make_scheduler(optimizer, CONFIG["epochs"] - epoch - 1)
                patience_counter = 0
                freeze_epochs = epoch + 1
            else:
                print(f"\nEarly stopping at epoch {epoch + 1} (patience={patience})")
                break

    print(f"\nTraining complete | best damage_F1={best_f1:.4f} | best mIoU={best_miou:.4f}")
else:
    print("Training is disabled. Set CONFIG['run_training'] = True to train.")
    print(f"Run directory: {CONFIG['run_dir']}")


In [ ]:
# ============================================================
# 9. Test evaluation and prediction visualization
# ============================================================

def load_best_model_if_available():
    checkpoint_path = Path(CONFIG["best_checkpoint_path"])
    if checkpoint_path.exists():
        checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(checkpoint["model_state_dict"])
        print(f"Loaded best checkpoint from {checkpoint_path}")
        best_f1 = checkpoint.get("best_damage_f1", -1.0)
        best_miou = checkpoint.get("best_miou", -1.0)
        print(f"  best_damage_f1={best_f1:.4f}, best_miou={best_miou:.4f}")
        return True
    return False


def plot_confusion_matrix(conf_matrix):
    norm_conf = conf_matrix.float()
    row_sums = norm_conf.sum(dim=1, keepdim=True).clamp_min(1.0)
    norm_conf = norm_conf / row_sums

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].imshow(conf_matrix.numpy(), cmap="Blues")
    axes[0].set_xticks(range(CONFIG["num_classes"]))
    axes[0].set_xticklabels([DAMAGE_CLASSES[i] for i in range(CONFIG["num_classes"])], rotation=45, ha="right")
    axes[0].set_yticks(range(CONFIG["num_classes"]))
    axes[0].set_yticklabels([DAMAGE_CLASSES[i] for i in range(CONFIG["num_classes"])])
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("True")
    axes[0].set_title("Confusion Matrix (counts)")

    im = axes[1].imshow(norm_conf.numpy(), cmap="Blues", vmin=0, vmax=1)
    axes[1].set_xticks(range(CONFIG["num_classes"]))
    axes[1].set_xticklabels([DAMAGE_CLASSES[i] for i in range(CONFIG["num_classes"])], rotation=45, ha="right")
    axes[1].set_yticks(range(CONFIG["num_classes"]))
    axes[1].set_yticklabels([DAMAGE_CLASSES[i] for i in range(CONFIG["num_classes"])])
    axes[1].set_xlabel("Predicted")
    axes[1].set_ylabel("True")
    axes[1].set_title("Normalized Confusion Matrix")
    plt.colorbar(im, ax=axes[1])

    plt.tight_layout()
    plt.show()


def show_prediction(dataset, index):
    model.eval()
    sample = dataset[index]
    pre_img = sample["pre_image"].unsqueeze(0).to(DEVICE)
    post_img = sample["post_image"].unsqueeze(0).to(DEVICE)
    true_mask = sample["post_mask"].cpu().numpy()

    with torch.no_grad():
        pred_mask = model(pre_img, post_img).argmax(dim=1).squeeze(0).cpu().numpy()

    error_map = (pred_mask != true_mask).astype(np.uint8)

    fig, axes = plt.subplots(1, 5, figsize=(22, 5))
    axes[0].imshow(denormalize_image(sample["pre_image"]))
    axes[0].set_title(f"Pre image\n{sample['sample_id']}")
    axes[0].axis("off")

    axes[1].imshow(denormalize_image(sample["post_image"]))
    axes[1].set_title("Post image")
    axes[1].axis("off")

    axes[2].imshow(true_mask, cmap=DAMAGE_CMAP, vmin=0, vmax=4)
    axes[2].set_title("Ground truth")
    axes[2].axis("off")

    axes[3].imshow(pred_mask, cmap=DAMAGE_CMAP, vmin=0, vmax=4)
    axes[3].set_title("Prediction")
    axes[3].axis("off")

    axes[4].imshow(error_map, cmap="magma")
    axes[4].set_title("Error map")
    axes[4].axis("off")

    fig.legend(handles=DAMAGE_LEGEND, loc="lower center", ncol=5, frameon=False)
    plt.tight_layout(rect=(0, 0.05, 1, 1))
    plt.show()


if load_best_model_if_available():
    test_metrics = validate_one_epoch(model, test_loader, criterion)

    print("=" * 65)
    print("FINAL PERFORMANCE REPORT")
    print("=" * 65)
    print(f"Test loss:            {test_metrics['loss']:.4f}")
    print(f"Test pixel accuracy:  {test_metrics['pixel_accuracy']:.4f}")
    print(f"Test mean IoU:        {test_metrics['miou']:.4f}")
    print(f"Test mean Dice:       {test_metrics['mean_dice']:.4f}")
    print(f"Test damage macro F1: {test_metrics['damage_macro_f1']:.4f}  << PRIMARY METRIC")
    print()
    print("Per-class breakdown:")
    print(f"  {'Class':<12} {'IoU':>8} {'Dice':>8} {'F1':>8}")
    print(f"  {'-'*12} {'-'*8} {'-'*8} {'-'*8}")
    for i in range(CONFIG["num_classes"]):
        iou = float(test_metrics["per_class_iou"][i])
        dice = float(test_metrics["per_class_dice"][i])
        f1 = float(test_metrics["per_class_f1"][i])
        print(f"  {DAMAGE_CLASSES[i]:<12} {iou:>8.4f} {dice:>8.4f} {f1:>8.4f}")

    plot_confusion_matrix(test_metrics["confusion_matrix"])

    example_count = min(CONFIG["show_examples"], len(test_dataset))
    example_indices = np.linspace(0, len(test_dataset) - 1, num=example_count, dtype=int)
    for idx in example_indices:
        show_prediction(test_dataset, int(idx))
else:
    print("No checkpoint found yet. Train first, then run this cell again.")
    print(f"Expected best checkpoint at: {CONFIG['best_checkpoint_path']}")
